In [ ]:
# ini gada nilainya 
import os
import matplotlib.pyplot as plt
import keras
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [ ]:
# ini cell bobot 10% kalo typo jadi 1/2
# spill kategori lebih dari 5
UKURAN = 250
data_path = "Rice"
kelas = sorted([k for k in os.listdir(data_path)])
print(kelas)

x, y = [], []
for label, nama_kelas in enumerate(kelas):
    folder = os.path.join(data_path, nama_kelas)
    print(folder)
    for nama_file in os.listdir(folder):
        if nama_file.endswith("Thumbs.db"): # karena di pc binus tiba2 ada, bisa hapus juga boleh 
            continue
        gambar = keras.utils.load_img(os.path.join(folder, nama_file), color_mode="grayscale", target_size=(UKURAN, UKURAN))

        x.append(np.array(gambar))
        y.append(label)

x = np.array(x)
y = np.array(y)

['Arborio', 'Basmati', 'Ipsala', 'Jasmine', 'Karacadag']
Rice\Arborio
Rice\Basmati
Rice\Ipsala
Rice\Jasmine
Rice\Karacadag


In [6]:
# Lihat hasil
print("Jumlah gambar: ", len(x))
print("Ukuran:", x.shape)
print("Ukuran:", y.shape)

Jumlah gambar:  500
Ukuran: (500, 250, 250)
Ukuran: (500,)


In [ ]:
# ini bobot preprocessing 10%
# one hot encoding
x = x.astype('float32') / 255.0 # wajib dilakukan
y = keras.utils.to_categorical(y, num_classes=len(kelas)) # ini one hot encoding

print(x.shape)
print(y[0])

(500, 250, 250)
[[0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0.]]


In [ ]:
# ini bobotnya cuma 55
# 80% Train, 20% temporary
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.2, random_state=42)

# 20% Temp -> 10% validation, 10% test
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42)

print("Train:", len(x_train))
print("Validation:", len(x_val))
print("Test:", len(x_test))

# pake stratify malah error

Train: 400
Validation: 50
Test: 50


In [ ]:
# DCNN
# mainin di sini aja, biar hasilnya ok

model = keras.Sequential([
    # input layer, bobot 5%
    keras.layers.Input(shape=(UKURAN, UKURAN, 1)),
    
    # Hidden layer: Conv -> Pool -> Conv -> Conv -> Pool -> Flatten, bobot 20%
    keras.layers.Conv2D(32, (3, 3), activation="relu"), # conv2d semakin besar
    keras.layers.MaxPooling2D(pool_size=(2, 2), strides=2), # biasanya 2*2 atau 3*3
    
    # harus dibuat conv2d lebihbesar, agar analisa yang dilakukan semakin detail
    keras.layers.Conv2D(64, (3, 3), activation="relu"), # ini range nya 2,2 sampai 5,5
    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2), strides=2),
    
    keras.layers.Flatten(),

    # output, bobot 10%
    keras.layers.Dense(128, activation="relu"), # kalau dense semakin kecil
    keras.layers.Dropout(0.3), # ini untuk menghindari overfit, cuma bisa di taro di dense
    keras.layers.Dense(64, activation="relu"), # 64 jumlah neuron
    keras.layers.Dropout(0.3),
    keras.layers.Dense(len(kelas), activation="softmax") # kalo di soal dibuat multiple bikin lebih dari 1
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 248, 248, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 124, 124, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 122, 122, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 120, 120, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 60, 60, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 230400)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │    29,491,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,555,653 (112.75 MB)

 Trainable params: 29,555,653 (112.75 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# ini harus pake gpu, udh aman harusnya training model, bobot 20%
# ini yg paling beda di soal nnti, sesuain soal
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
history = model.fit(x_train, y_train, batch_size=100, epochs=20, validation_data=(x_val, y_val))

In [ ]:
# bobotnya 15%
# suruh model tebak
prediksi = model.predict(x_test)
tebakan = np.argmax(prediksi, axis=1)
real = np.argmax(y_test, axis=1)

plt.figure(figsize=(15, 15))
for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(x_test[i], cmap = "gray")
    plt.title(f"Tebakan:{kelas[tebakan[i]]}\n Asli: {kelas[real[i]]}")

plt.show()

In [ ]:
# accuracy, ini juga termasuk predict masuk yang 15%
loss, accuracy = model.evaluate(x_test, y_test)
print("Loss di data test: ", loss)
print("Akurasi di data test: ", accuracy)

In [ ]:
# plotting nilainya 10%
# langkah terakhir, plotting grafik

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label="train acc")
plt.plot(history.history['val_accuracy'], label="val acc")
plt.xlabel("epochs")
plt.ylabel("acc")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label="train loss")
plt.plot(history.history['val_loss'], label="val loss")
plt.xlabel("epochs")
plt.ylabel("loss")
plt.legend()

plt.show()